In [20]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [21]:
dataset_file_path = '../dataset/Gross Profit vs Competitor 0514.xlsx'

df = pd.read_excel(dataset_file_path)

if df is not None:
  print(f"File {dataset_file_path} loaded successfully.")
else:
  print(f"Failed to load file {dataset_file_path}.")

File ../dataset/Gross Profit vs Competitor 0514.xlsx loaded successfully.


In [22]:
df.columns

Index(['Quote ID', 'Product', 'Kw', 'Unit Price', 'Qty', 'Subtotal Price',
       'Gross Margin Rate ', 'Convert to Order (0:Success, 1:Fail)',
       'Energy grant amount', 'Competitor A', 'Competitor B', 'Competitor C'],
      dtype='object')

# standarize column names

In [23]:
df.columns = [col.strip() for col in df.columns]
df.columns = [col.replace(' ', '_') for col in df.columns]
df.columns = [col.lower() for col in df.columns]

col_mapping = {
    "convert_to_order_(0:success,_1:fail)": "convert_to_order"
}
df = df.rename(columns=col_mapping)
df.columns

Index(['quote_id', 'product', 'kw', 'unit_price', 'qty', 'subtotal_price',
       'gross_margin_rate', 'convert_to_order', 'energy_grant_amount',
       'competitor_a', 'competitor_b', 'competitor_c'],
      dtype='object')

# data type

In [24]:
df.dtypes

quote_id                object
product                 object
kw                     float64
unit_price             float64
qty                      int64
subtotal_price         float64
gross_margin_rate      float64
convert_to_order         int64
energy_grant_amount      int64
competitor_a           float64
competitor_b           float64
competitor_c           float64
dtype: object

# drop qty <= 0

In [25]:
print(f"Size before drop: {df.shape}")
df = df[df['qty'] > 0]
print(f"Size after drop: {df.shape}")

Size before drop: (4994, 12)
Size after drop: (4991, 12)


# deduplication

## exact duplication

In [26]:
print(f"Size before drop exact duplicates: {df.shape}")
df.drop_duplicates(inplace=True)
print(f"Size after drop exact duplicates: {df.shape}")

Size before drop exact duplicates: (4991, 12)
Size after drop exact duplicates: (4902, 12)


## same quote id, same product id, but different price (only flag)

In [27]:
df = (
    df.sort_values(['quote_id', 'product', 'unit_price'])
      .assign(
          price_order=lambda x: x.groupby(['quote_id', 'product']).cumcount() + 1,
          is_highest_price=lambda x: (
              x['unit_price'] == x.groupby(['quote_id', 'product'])['unit_price'].transform('max')
          ).astype(int)
      )
)

same_quote_product = df[
    df.groupby(['quote_id', 'product'])['quote_id']
         .transform('count') > 1
]

same_quote_product[
    ['quote_id', 'product', 'kw', 'unit_price',
     'price_order', 'is_highest_price',
     'convert_to_order']
].sort_values(['quote_id', 'product', 'price_order'])

,quote_id,product,kw,unit_price,price_order,is_highest_price,convert_to_order
21,Q-01366,N1,75.0,683400.0,1,0,0
13,Q-01366,N1,75.0,845960.0,2,1,0
60,Q-02428,O,75.0,614000.0,1,0,0
26,Q-02428,O,75.0,659000.0,2,1,0
59,Q-02428,S,110.0,1084000.0,1,0,0
...,...,...,...,...,...,...,...
4875,Q-44580,R,7.5,98644.0,2,1,0
4876,Q-44580,X,11.0,118531.7,1,1,0
4877,Q-44580,X,11.0,118531.7,2,1,0
4924,Q-45260,X,11.0,123424.0,1,0,0


In [28]:
print(f"Size before filtering to highest price: {df.shape}")
df = df[df['is_highest_price'] == 1]
print(f"Size after filtering to highest price: {df.shape}")

Size before filtering to highest price: (4902, 14)
Size after filtering to highest price: (4833, 14)


# create competitor variable

In [29]:
competitor_cols = ["competitor_a", "competitor_b", "competitor_c"]

df['is_compe_a'] = df['competitor_a'].notna()
df['is_compe_b'] = df['competitor_b'].notna()
df['is_compe_c'] = df['competitor_c'].notna()

df['known_num_compe'] = df[['is_compe_a', 'is_compe_b', 'is_compe_c']].sum(axis=1)

df['avg_compe_price'] = df[competitor_cols].mean(axis=1)
df['max_compe_price'] = df[competitor_cols].max(axis=1)
df['min_compe_price'] = df[competitor_cols].min(axis=1)
df

,quote_id,product,kw,unit_price,qty,subtotal_price,gross_margin_rate,convert_to_order,energy_grant_amount,competitor_a,...,competitor_c,price_order,is_highest_price,is_compe_a,is_compe_b,is_compe_c,known_num_compe,avg_compe_price,max_compe_price,min_compe_price
0,Q-00114,W,11.0,136000.0,1,136000.0,0.0712,0,66000,NaN,...,NaN,1,1,False,False,False,0,NaN,NaN,NaN
1,Q-00119,H1,37.0,290000.0,2,580000.0,0.3079,0,222000,360000.0,...,NaN,1,1,True,False,False,1,360000.0,360000.0,360000.0
2,Q-00161,Y,150.0,2379000.0,1,2379000.0,0.4224,1,720000,NaN,...,NaN,1,1,False,False,False,0,NaN,NaN,NaN
8,Q-00167,O1,75.0,775970.0,1,775970.0,0.4117,0,375000,760000.0,...,NaN,1,1,True,True,False,2,830000.0,900000.0,760000.0
3,Q-00214,I1,22.0,300000.0,1,300000.0,0.6787,0,132000,240000.0,...,NaN,1,1,True,False,False,1,240000.0,240000.0,240000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4978,Q-45799,H1,37.0,373248.0,1,373248.0,0.4595,1,222000,360000.0,...,NaN,1,1,True,False,False,1,360000.0,360000.0,360000.0
4979,Q-45803,T,110.0,1590300.0,1,1590300.0,0.3578,1,528000,1880000.0,...,NaN,1,1,True,False,False,1,1880000.0,1880000.0,1880000.0
4981,Q-45808,C,110.0,1100000.0,2,2200000.0,0.1439,0,451000,NaN,...,NaN,1,1,False,False,False,0,NaN,NaN,NaN
4982,Q-45808,U,110.0,1500000.0,1,1500000.0,0.1439,0,528000,1880000.0,...,NaN,1,1,True,False,False,1,1880000.0,1880000.0,1880000.0


# price gap variables

In [30]:
df["competitor_count_available"] = df[competitor_cols].notna().sum(axis=1)

df["avg_competitor_price"] = df[competitor_cols].mean(axis=1)
df["min_competitor_price"] = df[competitor_cols].min(axis=1)
df["max_competitor_price"] = df[competitor_cols].max(axis=1)

df["price_gap_avg_competitor"] = (
    df["unit_price"] - df["avg_competitor_price"]
)

df["price_gap_avg_competitor_pct"] = (
    df["price_gap_avg_competitor"] / df["avg_competitor_price"]
) * 100

df["price_gap_min_competitor"] = (
    df["unit_price"] - df["min_competitor_price"]
)

df["price_gap_min_competitor_pct"] = (
    df["price_gap_min_competitor"] / df["min_competitor_price"]
) * 100

df["higher_than_avg_competitor"] = np.where(
    df["unit_price"] > df["avg_competitor_price"],
    1,
    0
)

df.loc[df["avg_competitor_price"].isna(), "higher_than_avg_competitor"] = np.nan

display(df[[
    "unit_price",
    "competitor_a",
    "competitor_b",
    "competitor_c",
    "competitor_count_available",
    "avg_competitor_price",
    "price_gap_avg_competitor_pct",
    "higher_than_avg_competitor",
    "convert_to_order"
]].head(20))

,unit_price,competitor_a,competitor_b,competitor_c,competitor_count_available,avg_competitor_price,price_gap_avg_competitor_pct,higher_than_avg_competitor,convert_to_order
0,136000.0,NaN,NaN,NaN,0,NaN,NaN,NaN,0
1,290000.0,360000.0,NaN,NaN,1,360000.0,-19.444444,0.0,0
2,2379000.0,NaN,NaN,NaN,0,NaN,NaN,NaN,1
8,775970.0,760000.0,900000.0,NaN,2,830000.0,-6.509639,0.0,0
3,300000.0,240000.0,NaN,NaN,1,240000.0,25.000000,1.0,0
11,400000.0,360000.0,NaN,NaN,1,360000.0,11.111111,1.0,0
4,228964.0,240000.0,NaN,NaN,1,240000.0,-4.598333,0.0,0
5,1283500.0,760000.0,900000.0,NaN,2,830000.0,54.638554,1.0,1
7,290000.0,360000.0,NaN,NaN,1,360000.0,-19.444444,0.0,0
6,270000.0,240000.0,NaN,NaN,1,240000.0,12.500000,1.0,0


In [31]:
df['is_lower_than_competitor'] = df['unit_price'] <= df['avg_competitor_price']
df

,quote_id,product,kw,unit_price,qty,subtotal_price,gross_margin_rate,convert_to_order,energy_grant_amount,competitor_a,...,competitor_count_available,avg_competitor_price,min_competitor_price,max_competitor_price,price_gap_avg_competitor,price_gap_avg_competitor_pct,price_gap_min_competitor,price_gap_min_competitor_pct,higher_than_avg_competitor,is_lower_than_competitor
0,Q-00114,W,11.0,136000.0,1,136000.0,0.0712,0,66000,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,Q-00119,H1,37.0,290000.0,2,580000.0,0.3079,0,222000,360000.0,...,1,360000.0,360000.0,360000.0,-70000.0,-19.444444,-70000.0,-19.444444,0.0,True
2,Q-00161,Y,150.0,2379000.0,1,2379000.0,0.4224,1,720000,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
8,Q-00167,O1,75.0,775970.0,1,775970.0,0.4117,0,375000,760000.0,...,2,830000.0,760000.0,900000.0,-54030.0,-6.509639,15970.0,2.101316,0.0,True
3,Q-00214,I1,22.0,300000.0,1,300000.0,0.6787,0,132000,240000.0,...,1,240000.0,240000.0,240000.0,60000.0,25.000000,60000.0,25.000000,1.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4978,Q-45799,H1,37.0,373248.0,1,373248.0,0.4595,1,222000,360000.0,...,1,360000.0,360000.0,360000.0,13248.0,3.680000,13248.0,3.680000,1.0,False
4979,Q-45803,T,110.0,1590300.0,1,1590300.0,0.3578,1,528000,1880000.0,...,1,1880000.0,1880000.0,1880000.0,-289700.0,-15.409574,-289700.0,-15.409574,0.0,True
4981,Q-45808,C,110.0,1100000.0,2,2200000.0,0.1439,0,451000,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4982,Q-45808,U,110.0,1500000.0,1,1500000.0,0.1439,0,528000,1880000.0,...,1,1880000.0,1880000.0,1880000.0,-380000.0,-20.212766,-380000.0,-20.212766,0.0,True


# feature engineering: perhitungan

In [32]:
df['effective_price_after_grant'] = (
    df['subtotal_price'] - df['energy_grant_amount']
)

df['grant_ratio_to_subtotal'] = (
    df['energy_grant_amount'] / df['subtotal_price']
)

In [33]:
df['estimated_cost'] = (
    df['subtotal_price'] * (1 - df['gross_margin_rate'])
)

df['estimated_gross_profit'] = (
    df['subtotal_price'] - df['estimated_cost']
)

# save csv file

In [34]:
df.to_csv('../dataset/df_preprocessed.csv', index=False)

In [35]:
df.shape

(4833, 35)

In [36]:
df.describe()

,kw,unit_price,qty,subtotal_price,gross_margin_rate,convert_to_order,energy_grant_amount,competitor_a,competitor_b,competitor_c,...,max_competitor_price,price_gap_avg_competitor,price_gap_avg_competitor_pct,price_gap_min_competitor,price_gap_min_competitor_pct,higher_than_avg_competitor,effective_price_after_grant,grant_ratio_to_subtotal,estimated_cost,estimated_gross_profit
count,4833.000000,4.833000e+03,4833.000000,4.833000e+03,4833.000000,4833.000000,4833.000000,3.969000e+03,445.0,5.500000e+01,...,4.024000e+03,4.024000e+03,4024.000000,4.024000e+03,4024.000000,4024.000000,4.833000e+03,4833.000000,4.833000e+03,4.833000e+03
mean,33.498552,4.341185e+05,1.165322,4.977244e+05,0.377900,0.767846,181553.105731,3.560850e+05,900000.0,2.009091e+06,...,3.941604e+05,5.890466e+04,27.149235,6.664571e+04,28.065121,0.742793,3.161713e+05,0.425643,2.913701e+05,2.063543e+05
std,30.552675,4.702373e+05,0.788854,6.135237e+05,0.175544,0.422250,140875.931733,2.806387e+05,0.0,2.901294e+04,...,3.581574e+05,2.298748e+05,99.403402,2.260900e+05,99.148560,0.437149,5.130282e+05,0.153207,3.731846e+05,3.013446e+05
min,7.500000,4.020000e+04,1.000000,4.020000e+04,-1.322600,0.000000,38700.000000,1.050000e+04,900000.0,2.000000e+06,...,1.050000e+04,-9.275000e+05,-53.443373,-9.275000e+05,-49.155263,0.000000,-3.692000e+04,0.031426,3.878496e+04,-4.236819e+05
25%,15.000000,2.290000e+05,1.000000,2.369250e+05,0.262200,1.000000,90000.000000,2.400000e+05,900000.0,2.000000e+06,...,2.400000e+05,-4.248000e+03,-1.180000,1.584000e+03,0.440000,0.000000,1.249320e+05,0.313390,1.582874e+05,7.378770e+04
50%,22.000000,2.859480e+05,1.000000,3.261500e+05,0.400100,1.000000,132000.000000,2.400000e+05,900000.0,2.000000e+06,...,2.400000e+05,2.750400e+04,10.565000,2.785760e+04,10.565000,1.000000,1.478640e+05,0.392370,1.614443e+05,1.291347e+05
75%,37.000000,4.212000e+05,1.000000,5.307120e+05,0.478200,1.000000,222000.000000,3.600000e+05,900000.0,2.000000e+06,...,3.600000e+05,7.000000e+04,36.985294,7.057000e+04,37.368421,1.000000,3.036000e+05,0.548971,2.032767e+05,2.361689e+05
max,200.000000,4.600000e+06,16.000000,1.044320e+07,0.921100,1.000000,960000.000000,1.880000e+06,900000.0,2.100000e+06,...,2.100000e+06,2.600000e+06,1804.761905,2.600000e+06,1804.761905,1.000000,9.723200e+06,1.239740,6.854421e+06,5.086883e+06


In [37]:
df.describe(include='object')

,quote_id,product
count,4833,4833
unique,4640,43
top,Q-11450,I1
freq,4,1482


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4833 entries, 0 to 4980
Data columns (total 35 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   quote_id                      4833 non-null   object 
 1   product                       4833 non-null   object 
 2   kw                            4833 non-null   float64
 3   unit_price                    4833 non-null   float64
 4   qty                           4833 non-null   int64  
 5   subtotal_price                4833 non-null   float64
 6   gross_margin_rate             4833 non-null   float64
 7   convert_to_order              4833 non-null   int64  
 8   energy_grant_amount           4833 non-null   int64  
 9   competitor_a                  3969 non-null   float64
 10  competitor_b                  445 non-null    float64
 11  competitor_c                  55 non-null     float64
 12  price_order                   4833 non-null   int64  
 13  is_highe

In [39]:
df.columns

Index(['quote_id', 'product', 'kw', 'unit_price', 'qty', 'subtotal_price',
       'gross_margin_rate', 'convert_to_order', 'energy_grant_amount',
       'competitor_a', 'competitor_b', 'competitor_c', 'price_order',
       'is_highest_price', 'is_compe_a', 'is_compe_b', 'is_compe_c',
       'known_num_compe', 'avg_compe_price', 'max_compe_price',
       'min_compe_price', 'competitor_count_available', 'avg_competitor_price',
       'min_competitor_price', 'max_competitor_price',
       'price_gap_avg_competitor', 'price_gap_avg_competitor_pct',
       'price_gap_min_competitor', 'price_gap_min_competitor_pct',
       'higher_than_avg_competitor', 'is_lower_than_competitor',
       'effective_price_after_grant', 'grant_ratio_to_subtotal',
       'estimated_cost', 'estimated_gross_profit'],
      dtype='object')